# Models: predicting the DPE class of a Paris dwelling
**MSA-DATI07-03** · Group: Elie Maarek, Joshai Bensoussan, Lucas Noury

We start from the clean data of notebook 01 (`data/processed/clean.parquet`). We compare three models: a logistic regression with gradient descent coded by hand, a decision tree and a random forest.

Three important choices, explained below:
- we **exclude** the columns that compute the label (consumption, GHG), otherwise the model cheats;
- we **group** the classes so that each one has enough dwellings;
- we split train and test **by building**, because dwellings of the same building look alike.

In [ ]:
# --- Parameters ---
SEED = 42
DATA_DIR = "../data"
REPORT_DIR = "../reports"
TEST_SIZE = 0.2                                   # share of buildings in the test set
GROUPED_CLASSES = {"A": "A-B", "B": "A-B", "C": "C", "D": "D", "E": "E", "F": "F-G", "G": "F-G"}
LEAKAGE = ["conso_5_usages_par_m2_ep", "emission_ges_5_usages_par_m2", "etiquette_ges", "conso_outlier"]
MIN_CATEGORY = 50                                 # categories with fewer dwellings in train go to "other"
LR, N_ITER, L2 = 1.0, 4000, 1e-4                  # hand-coded logistic regression
DEPTHS = [3, 4, 5, 6, 8, 10, 12, 15]                      # decision tree depths tried
N_FOLDS = 5
N_TREES, MIN_LEAF = 300, 2                        # random forest

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
sns.set_theme(style="whitegrid")

df = pd.read_parquet(DATA / "processed" / "clean.parquet")
print(df.shape)

## 1 · Target and variables

Classes A and B are only 1 % of dwellings, so we group them as "A-B", and F and G as "F-G". This leaves 5 classes.

In [ ]:
classes = ["A-B", "C", "D", "E", "F-G"]
y = df["etiquette_dpe"].map(GROUPED_CLASSES).map({c: i for i, c in enumerate(classes)}).to_numpy()
print(pd.Series(y).map(dict(enumerate(classes))).value_counts(normalize=True)[classes].round(3))

Two sets of variables:
- **descriptors**: surface, construction period, house or apartment, DPE generated from the building, heating, postal code;
- **descriptors + intermediate variables**: we add heat losses, ubat and insulation quality, which are results of calculations on the building (but not on consumption).

We exclude `conso_5_usages_par_m2_ep`, `emission_ges_5_usages_par_m2`, `etiquette_ges` and `conso_outlier` (list `LEAKAGE`), because they come from the same calculation as the label.

In [ ]:
def order(p):
    if p.startswith("avant"):
        return 0
    if p.startswith("après"):
        return 9999
    return int(p[:4])

periods = sorted(df["periode_construction"].unique(), key=order)
insulation = {"insuffisante": 0, "moyenne": 1, "bonne": 2, "très bonne": 3}
CATEGORICAL = [("type_energie_principale_chauffage", "heating"),
               ("type_installation_chauffage", "install"),
               ("code_postal_ban", "zip")]


def prepare(train, test, intermediate):
    """Build X_train and X_test. Everything learned (medians, categories) comes from train only."""
    def base(d, cats):
        X = pd.DataFrame(index=d.index)
        X["surface"] = d["surface_habitable_logement"]
        X["period"] = d["periode_construction"].map({p: i for i, p in enumerate(periods)})
        X["house"] = (d["type_batiment"] == "maison").astype(int)
        X["dpe_immeuble_genere"] = d["dpe_immeuble_genere"].astype(int)
        for col, name in CATEGORICAL:
            v = d[col].fillna("unknown")
            v = v.where(v.isin(cats[col]), "other")
            X = X.join(pd.get_dummies(pd.Categorical(v, categories=cats[col] + ["other"]),
                                      prefix=name, dtype=int).set_index(d.index))
        if intermediate:
            X["log_heat_loss"] = np.log1p(d["deperditions_enveloppe"])
            X["ubat"] = d["ubat_w_par_m2_k"]
            X["insulation"] = d["qualite_isolation_enveloppe"].map(insulation)
        return X

    cats = {}
    for col, _ in CATEGORICAL:
        v = train[col].fillna("unknown").value_counts()
        cats[col] = list(v[v >= MIN_CATEGORY].index)
    Xtr, Xte = base(train, cats), base(test, cats)
    med = Xtr.median()
    return Xtr.fillna(med), Xte.fillna(med)

## 2 · Train / test split by building

Dwellings of the same building often have almost identical values (especially when their DPE is generated from the building). With a random split, a test dwelling would have a "twin" in train and the scores would be too good. So we split by `identifiant_ban`.

In [ ]:
groups = df["identifiant_ban"].fillna(df["numero_dpe"])          # dwellings without a building: one group each
idx_train, idx_test = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(df, y, groups))
train, test = df.iloc[idx_train], df.iloc[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
print(len(train), "train ·", len(test), "test ·", len(set(groups.iloc[idx_train]) & set(groups.iloc[idx_test])), "buildings in common")

## 3 · Logistic regression coded by hand

Softmax model (logistic regression with 5 classes) trained by gradient descent on the whole train set. Variables are standardised with the mean and standard deviation of train. We check the result against scikit-learn.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


class LogisticRegressionGD:
    def __init__(self, lr=LR, n_iter=N_ITER, l2=L2):
        self.lr, self.n_iter, self.l2 = lr, n_iter, l2

    def fit(self, X, y, k):
        n, d = X.shape
        self.W, self.b = np.zeros((d, k)), np.zeros(k)
        Y = np.eye(k)[y]
        self.losses = []
        for _ in range(self.n_iter):
            P = softmax(X @ self.W + self.b)
            self.losses.append(-np.mean(np.log(P[np.arange(n), y] + 1e-12)))
            G = P - Y
            self.W -= self.lr * (X.T @ G / n + self.l2 * self.W)
            self.b -= self.lr * G.mean(axis=0)
        return self

    def predict(self, X):
        return np.argmax(X @ self.W + self.b, axis=1)

In [ ]:
predictions = {}      # (variable set, model) -> predictions on test
losses = {}

for name, interm in [("descriptors", False), ("descriptors + intermediate", True)]:
    Xtr, Xte = prepare(train, test, interm)
    mu, sd = Xtr.mean(), Xtr.std().replace(0, 1)
    Ztr, Zte = ((Xtr - mu) / sd).to_numpy(), ((Xte - mu) / sd).to_numpy()
    model = LogisticRegressionGD().fit(Ztr, y_train, len(classes))
    predictions[(name, "logistic regression (by hand)")] = model.predict(Zte)
    losses[name] = model.losses
    # check: scikit-learn should give about the same
    ref = LogisticRegression(max_iter=3000, C=1000).fit(Ztr, y_train)
    print(name, "· accuracy by hand:", round(accuracy_score(y_test, model.predict(Zte)), 3),
          "· scikit-learn:", round(accuracy_score(y_test, ref.predict(Zte)), 3))

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, l in losses.items():
    ax.plot(l, label=name)
ax.set_xlabel("iteration")
ax.set_ylabel("loss (cross-entropy)")
ax.legend()
fig.suptitle("Gradient descent converges", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(REPORTS / "6_1_loss.png", dpi=120, bbox_inches="tight")
plt.show()

## 4 · Decision tree and random forest

For the tree, we choose the depth by cross-validation **on train**, with folds also split by building. Test is only used at the end.

In [ ]:
cv = GroupKFold(n_splits=N_FOLDS)
g_train = groups.iloc[idx_train].to_numpy()

for name, interm in [("descriptors", False), ("descriptors + intermediate", True)]:
    Xtr, Xte = prepare(train, test, interm)
    scores = []
    for depth in DEPTHS:
        f1s = []
        for a, b in cv.split(Xtr, y_train, g_train):
            m = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(Xtr.iloc[a], y_train[a])
            f1s.append(f1_score(y_train[b], m.predict(Xtr.iloc[b]), average="macro"))
        scores.append(np.mean(f1s))
    best_depth = DEPTHS[int(np.argmax(scores))]
    print(name, "· chosen depth:", best_depth, "· validation macro F1:", round(max(scores), 3))
    tree = DecisionTreeClassifier(max_depth=best_depth, random_state=SEED).fit(Xtr, y_train)
    forest = RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=MIN_LEAF, random_state=SEED, n_jobs=-1).fit(Xtr, y_train)
    predictions[(name, "decision tree")] = tree.predict(Xte)
    predictions[(name, "random forest")] = forest.predict(Xte)
    if interm:
        importances = pd.Series(forest.feature_importances_, index=Xtr.columns).sort_values()

## 5 · Comparison

We compare with a model that always predicts the most frequent class. Three measures: accuracy, macro F1 (mean of the F1 of each class, so small classes count as much as big ones) and the share of dwellings predicted within one class (classes are ordered).

In [ ]:
majority = np.bincount(y_train).argmax()
rows = [("none", "always the most frequent class", np.full(len(y_test), majority))]
rows += [(name, model, pred) for (name, model), pred in predictions.items()]
table = pd.DataFrame([{
    "variables": name, "model": model,
    "accuracy": accuracy_score(y_test, pred),
    "macro F1": f1_score(y_test, pred, average="macro"),
    "within one class": np.mean(np.abs(pred - y_test) <= 1),
} for name, model, pred in rows]).round(3)
table.to_csv(REPORTS / "model_comparison.csv", index=False)
table

In [ ]:
best = table.iloc[1:].sort_values("macro F1").iloc[-1]
pred = predictions[(best["variables"], best["model"])]
print("best model:", best["model"], "·", best["variables"])

cm = confusion_matrix(y_test, pred, normalize="true") * 100
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.1]})
sns.heatmap(cm, annot=True, fmt=".0f", cmap="Blues", xticklabels=classes, yticklabels=classes,
            cbar=False, ax=a1)
a1.set_xlabel("predicted class")
a1.set_ylabel("true class")
a1.set_title("% of each true class", fontsize=10)
importances.tail(10).plot.barh(ax=a2)
a2.set_title("10 most important variables (forest)", fontsize=10)
fig.suptitle(f"{best['model'].capitalize()}: accuracy {best['accuracy']:.1%}, errors are almost always within one class",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(REPORTS / "6_2_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

## 6 · Conclusion

**Results** (test set: 1,948 dwellings, 20 % of the buildings)
- Always predicting the most frequent class gives 31 % accuracy and 0.09 macro F1: this is the bar to beat.
- With **descriptors only**, all three models are close (accuracy 41 to 43 %, macro F1 0.35 to 0.37). The random forest is slightly ahead, but with a single split of 1,948 dwellings a gap of 1 or 2 points is not a real difference.
- With **intermediate variables** added, the scores jump: the random forest reaches 64.5 % accuracy and 0.56 macro F1, the decision tree 58 %, and our logistic regression 54 %. The prediction is within one class for 97 % of dwellings with the forest.
- The hand-coded logistic regression gives the same result as scikit-learn (accuracy 0.415 against 0.415 on descriptors, 0.536 against 0.542 with intermediate variables) and its loss decreases steadily (figure 6.1).
- The most important variables are heat loss, surface and ubat (figure 6.2).

**What cannot be trusted**
- Heat loss and ubat are computed from the building in the same calculation that gives the consumption, so the model with intermediate variables is **not** a prediction made before the diagnosis. The honest score for "predict the class from the description of the dwelling" is the descriptors-only one, about 43 %.
- Class A-B has only 36 dwellings in the test set and its recall is 8 %: the models almost never find it.
- One split, no confidence interval, and a sample of Paris DPE made for sales and rentals: the scores cannot be generalised to all dwellings.

**Next step:** test hypotheses H1 to H5 of notebook 01 with proper statistical tests, and look for descriptors that do better than 43 %.
